In [3]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from typing import TypedDict,Annotated,Literal
from pydantic import BaseModel,Field
from langchain_core.messages import SystemMessage, HumanMessage
import operator
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [19]:
llm1 = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",  # provider-backed
    task="text-generation",
    temperature=0.7,
)

llm2=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3",
    task="text-generation",
    temperature=0.3,
)

llm3=HuggingFaceEndpoint(
    repo_id="moonshotai/Kimi-K2-Instruct-0905",
    task="text-generation",
    temperature=0.7,
)


GeneratorModel = ChatHuggingFace(llm=llm3)
OptimizerModel=ChatHuggingFace(llm=llm3)
EvalutorModel=ChatHuggingFace(llm=llm1)

In [6]:
def generate_joke(state:JokeState):
    prompt = f"""You are a professional comedian with an exceptional sense of humor and creativity. 
Your task is to create the funniest, most clever joke on the following topic: {state['topic']}.
Make sure the joke is original, appropriate, and delivers a good punchline.
Keep it concise but impactful."""
    response=GeneratorModel.invoke(prompt)
    return {"joke":response.content}

In [7]:
def explain_joke(state:JokeState):
    prompt = f"""You are an expert humor analyst and comedian. 
Your task is to provide a clear, insightful explanation of why the following joke is funny and how it works.
Joke: {state['joke']}
Topic: {state['topic']}
Explain the setup, punchline, wordplay, or cultural references that make it humorous.
Keep the explanation concise but thorough."""
    
    response=EvalutorModel.invoke(prompt)
    return {"explanation":response.content}

In [23]:
graph=StateGraph(JokeState)

graph.add_node("generate_joke",generate_joke)
graph.add_node("explain_joke",explain_joke)

graph.add_edge(START,"generate_joke")
graph.add_edge('generate_joke',"explain_joke")
graph.add_edge("explain_joke",END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [24]:
config1={"configurable":{"thread_id":"1"}}
final_state=workflow.invoke({"topic":"Mosquito"},config=config1)
print(final_state)

{'topic': 'Mosquito', 'joke': 'Mosquitoes have the world’s worst dating app: they swipe right, you swipe left, and somehow you still end up sharing bodily fluids.', 'explanation': 'Let\'s dissect this joke and understand why it\'s funny.\n\n**Setup:** The joke starts by establishing a relatable scenario – a dating app. This makes the listener (or reader) curious about the premise.\n\n**Punchline:** "you swipe left, and somehow you still end up sharing bodily fluids." This is where the humor kicks in. The phrase "sharing bodily fluids" typically implies a romantic or intimate encounter. The unexpected twist is that this sharing of bodily fluids happens despite the app\'s supposed purpose of avoiding such interactions (since you swiped left, indicating disinterest).\n\n**Wordplay:** The joke relies on a clever play on the typical dating app experience. Swiping right usually indicates interest, while swiping left means you\'re not interested. In this joke, the speaker uses the words "swip

In [26]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Mosquito', 'joke': 'Mosquitoes have the world’s worst dating app: they swipe right, you swipe left, and somehow you still end up sharing bodily fluids.', 'explanation': 'Let\'s dissect this joke and understand why it\'s funny.\n\n**Setup:** The joke starts by establishing a relatable scenario – a dating app. This makes the listener (or reader) curious about the premise.\n\n**Punchline:** "you swipe left, and somehow you still end up sharing bodily fluids." This is where the humor kicks in. The phrase "sharing bodily fluids" typically implies a romantic or intimate encounter. The unexpected twist is that this sharing of bodily fluids happens despite the app\'s supposed purpose of avoiding such interactions (since you swiped left, indicating disinterest).\n\n**Wordplay:** The joke relies on a clever play on the typical dating app experience. Swiping right usually indicates interest, while swiping left means you\'re not interested. In this joke, the speaker

In [27]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Mosquito', 'joke': 'Mosquitoes have the world’s worst dating app: they swipe right, you swipe left, and somehow you still end up sharing bodily fluids.', 'explanation': 'Let\'s dissect this joke and understand why it\'s funny.\n\n**Setup:** The joke starts by establishing a relatable scenario – a dating app. This makes the listener (or reader) curious about the premise.\n\n**Punchline:** "you swipe left, and somehow you still end up sharing bodily fluids." This is where the humor kicks in. The phrase "sharing bodily fluids" typically implies a romantic or intimate encounter. The unexpected twist is that this sharing of bodily fluids happens despite the app\'s supposed purpose of avoiding such interactions (since you swiped left, indicating disinterest).\n\n**Wordplay:** The joke relies on a clever play on the typical dating app experience. Swiping right usually indicates interest, while swiping left means you\'re not interested. In this joke, the speake